# GraphPad Prism Data Explorer

Parse `.prism` files and plot XY data with mean ± SEM across replicates.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from prism_parser import parse_prism

PRISM_FILE = "Prism_Data.prism"
sheets = parse_prism(PRISM_FILE)
print(f"Loaded {len(sheets)} sheets: {list(sheets.keys())}")

# ---------- Color scheme configuration ----------
# Choose a matplotlib colormap for the dose columns (e.g. -12 through -6).
# Good options: "viridis", "plasma", "inferno", "magma", "cividis",
#               "coolwarm", "RdYlBu", "Spectral", "turbo"
DOSE_CMAP = "plasma"

# Fixed colors for the control / reference datasets
COLOR_BUFFER = "grey"
COLOR_FSK = "black"

# ---------- Axis limits ----------
X_MIN = 0    # minutes
X_MAX = 30   # minutes


def get_color_map(dataset_names, cmap_name=DOSE_CMAP):
    """Return a dict mapping dataset name -> color.

    Numeric-looking dose labels (e.g. '-11', '-6') are spread across
    the chosen colormap.  'buffer' and '10uM Fsk' get fixed colors.
    """
    dose_names = [n for n in dataset_names if n not in ("buffer", "10uM Fsk")]
    dose_names.sort(key=lambda s: float(s))
    cmap = mpl.colormaps[cmap_name].resampled(max(len(dose_names), 1))
    colors = {}
    for i, name in enumerate(dose_names):
        colors[name] = cmap(i / max(len(dose_names) - 1, 1))
    colors["buffer"] = COLOR_BUFFER
    colors["10uM Fsk"] = COLOR_FSK
    return colors

## Plot all sheets — mean ± SEM per dataset

In [ ]:
for sheet_title, sheet in sheets.items():
    df = sheet.df
    x = df["X"].values
    y_data = df.drop(columns="X")
    dataset_names = y_data.columns.get_level_values("dataset").unique()
    colors = get_color_map(dataset_names)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax_fsk = ax.twinx()  # right y-axis for FSK

    for ds_name in dataset_names:
        ds_cols = y_data[ds_name]
        mean = ds_cols.mean(axis=1)
        sem = ds_cols.sem(axis=1)

        target_ax = ax_fsk if ds_name == "10uM Fsk" else ax
        target_ax.plot(x, mean, label=ds_name, color=colors[ds_name])
        target_ax.fill_between(x, mean - sem, mean + sem, alpha=0.1, color=colors[ds_name])

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_xlabel(sheet.xlabel)
    ax.set_ylabel(sheet.ylabel)
    ax_fsk.set_ylabel("10uM Fsk  " + sheet.ylabel, color=COLOR_FSK)
    ax_fsk.tick_params(axis="y", labelcolor=COLOR_FSK)
    ax.set_title(sheet.graph_title)
    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")


    

    # Combine legends from both axes
    handles_l, labels_l = ax.get_legend_handles_labels()
    handles_r, labels_r = ax_fsk.get_legend_handles_labels()
    ax.legend(handles_l + handles_r, labels_l + labels_r, loc="best", fontsize="small")

    plt.tight_layout()

    filename = sheet.graph_title.replace(" ", "_").replace("/", "-") + ".png"
    fig.savefig(filename, dpi=150)
    print(f"Saved {filename}")

    plt.show()


## Inspect a single sheet

In [ ]:
# Change the sheet name here to explore different datasets
SHEET = list(sheets.keys())[0]
sheet = sheets[SHEET]
df = sheet.df
print(f"Sheet: {SHEET}")
print(f"Graph title: {sheet.graph_title}")
print(f"X label: {sheet.xlabel}")
print(f"Y label: {sheet.ylabel}")
print(f"Shape: {df.shape}")
print(f"X range: {df['X'].min():.2f} – {df['X'].max():.2f}")
print(f"Datasets: {df.drop(columns='X').columns.get_level_values('dataset').unique().tolist()}")
df.head(10)